In [0]:
from pyspark.sql.functions import *

sat_customer = spark.table("default.sat_customer")

sat_customer = sat_customer \
.withColumn("Effective_From", col("Load_Date")) \
.withColumn("Effective_To", lit(None).cast("timestamp")) \
.withColumn("Is_Current", lit(True))

In [0]:
%sql
DROP TABLE IF EXISTS default.sat_customer;

In [0]:
sat_customer.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.sat_customer")

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import *

new_customer = [
    Row(
        Customer_ID="C001",
        Name="John Smith",
        Email="john@example.com",
        City="Hyderabad"
    ),
    Row(
        Customer_ID="C101",
        Name="Rahul",
        Email="rahul@gmail.com",
        City="Bangalore"
    )
]

incremental_df = spark.createDataFrame(new_customer) \
    .withColumn("Load_Date", current_timestamp()) \
    .withColumn("Record_Source", lit("Incremental")) \
    .withColumn(
        "HashDiff",
        sha2(
            concat_ws(
                "|",
                col("Name"),
                col("Email"),
                col("City")
            ),
            256
        )
    )

In [0]:
incremental_df.show(truncate=False)

+-----------+----------+----------------+---------+--------------------------+-------------+----------------------------------------------------------------+
|Customer_ID|Name      |Email           |City     |Load_Date                 |Record_Source|HashDiff                                                        |
+-----------+----------+----------------+---------+--------------------------+-------------+----------------------------------------------------------------+
|C001       |John Smith|john@example.com|Hyderabad|2026-07-13 13:58:26.214551|Incremental  |d86317901a2a5c524cd50d9901c40d495316615513cb78142c18d5ae79243142|
|C101       |Rahul     |rahul@gmail.com |Bangalore|2026-07-13 13:58:26.214551|Incremental  |a2cb560a693eceb5cee85f2b4fc06c745bf13338f37c170ad73b7b42e6a5bba1|
+-----------+----------+----------------+---------+--------------------------+-------------+----------------------------------------------------------------+



In [0]:
hub_customer = spark.table("default.hub_customer")
sat_customer = spark.table("default.sat_customer")

In [0]:
hub_customer = spark.table("default.hub_customer")
incremental_df.show()

+-----------+----------+----------------+---------+--------------------+-------------+--------------------+
|Customer_ID|      Name|           Email|     City|           Load_Date|Record_Source|            HashDiff|
+-----------+----------+----------------+---------+--------------------+-------------+--------------------+
|       C001|John Smith|john@example.com|Hyderabad|2026-07-13 14:02:...|  Incremental|d86317901a2a5c524...|
|       C101|     Rahul| rahul@gmail.com|Bangalore|2026-07-13 14:02:...|  Incremental|a2cb560a693eceb5c...|
+-----------+----------+----------------+---------+--------------------+-------------+--------------------+



In [0]:
from pyspark.sql.functions import col, lit

stg_customer = (
    incremental_df.alias("i")
    .join(
        hub_customer.alias("h"),
        col("i.Customer_ID") == col("h.Customer_ID"),
        "left"
    )
    .select(
        col("h.Customer_HK").alias("Customer_HK"),
        col("i.Customer_ID"),
        col("i.Name").alias("Customer_Name"),
        col("i.Email"),
        col("i.City"),
        col("i.HashDiff"),
        col("i.Load_Date").alias("Effective_From"),
        lit(None).cast("timestamp").alias("Effective_To"),
        lit(True).alias("Is_Current"),
        col("i.Record_Source")
    )
)

In [0]:
stg_customer.show(truncate=False)

+----------------------------------------------------------------+-----------+-------------+----------------+---------+----------------------------------------------------------------+--------------------------+------------+----------+-------------+
|Customer_HK                                                     |Customer_ID|Customer_Name|Email           |City     |HashDiff                                                        |Effective_From            |Effective_To|Is_Current|Record_Source|
+----------------------------------------------------------------+-----------+-------------+----------------+---------+----------------------------------------------------------------+--------------------------+------------+----------+-------------+
|a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|C001       |John Smith   |john@example.com|Hyderabad|d86317901a2a5c524cd50d9901c40d495316615513cb78142c18d5ae79243142|2026-07-13 14:04:55.859578|NULL        |true      |Incremental  |


In [0]:
current_sat = spark.table("default.sat_customer") \
    .filter(col("Is_Current") == True)

In [0]:
current_sat.show(5)

+--------------------+-------------+-------------------+------+--------------------+--------------------+-------------+--------------------+------------+----------+
|         Customer_HK|Customer_Name|              Email|  City|            HashDiff|           Load_Date|Record_Source|      Effective_From|Effective_To|Is_Current|
+--------------------+-------------+-------------------+------+--------------------+--------------------+-------------+--------------------+------------+----------+
|a4906a54b9b8416b9...|   Customer_1|customer1@gmail.com|City_2|8d7ef09c398a8fc4f...|2026-07-13 10:01:...| Customer_CSV|2026-07-13 10:01:...|        NULL|      true|
|fdc98571fc93e8482...|   Customer_2|customer2@gmail.com|City_3|5682cc4efd8ffa1d1...|2026-07-13 10:01:...| Customer_CSV|2026-07-13 10:01:...|        NULL|      true|
|64604487c74ef2fb0...|   Customer_3|customer3@gmail.com|City_4|add7d8c2cae5b6073...|2026-07-13 10:01:...| Customer_CSV|2026-07-13 10:01:...|        NULL|      true|
|f12ce5042

In [0]:
changed_records = (
    stg_customer.alias("s")
    .join(
        current_sat.alias("t"),
        col("s.Customer_HK") == col("t.Customer_HK"),
        "left"
    )
    .filter(
        (col("t.Customer_HK").isNull()) |
        (col("s.HashDiff") != col("t.HashDiff"))
    )
    .select("s.*")
)

In [0]:
changed_records.show(truncate=False)

+----------------------------------------------------------------+-----------+-------------+----------------+---------+----------------------------------------------------------------+--------------------------+------------+----------+-------------+
|Customer_HK                                                     |Customer_ID|Customer_Name|Email           |City     |HashDiff                                                        |Effective_From            |Effective_To|Is_Current|Record_Source|
+----------------------------------------------------------------+-----------+-------------+----------------+---------+----------------------------------------------------------------+--------------------------+------------+----------+-------------+
|a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|C001       |John Smith   |john@example.com|Hyderabad|d86317901a2a5c524cd50d9901c40d495316615513cb78142c18d5ae79243142|2026-07-13 14:05:58.820918|NULL        |true      |Incremental  |


In [0]:
records_to_close = (
    current_sat.alias("t")
    .join(
        changed_records.select("Customer_HK").alias("s"),
        col("t.Customer_HK") == col("s.Customer_HK"),
        "inner"
    )
    .select("t.*")
)

In [0]:
records_to_close.show(truncate=False)

+----------------------------------------------------------------+-------------+-------------------+------+----------------------------------------------------------------+--------------------------+-------------+--------------------------+------------+----------+
|Customer_HK                                                     |Customer_Name|Email              |City  |HashDiff                                                        |Load_Date                 |Record_Source|Effective_From            |Effective_To|Is_Current|
+----------------------------------------------------------------+-------------+-------------------+------+----------------------------------------------------------------+--------------------------+-------------+--------------------------+------------+----------+
|a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|Customer_1   |customer1@gmail.com|City_2|8d7ef09c398a8fc4f2214c4ca3b47d8b5ba929fe0b21145c0ddb8634570357cd|2026-07-13 10:01:07.445381|Custom

In [0]:
closed_records = records_to_close.withColumn(
    "Effective_To",
    current_timestamp()
).withColumn(
    "Is_Current",
    lit(False)
)

In [0]:
closed_records.select(
    "Customer_HK",
    "Customer_Name",
    "Is_Current",
    "Effective_From",
    "Effective_To"
).show(truncate=False)

+----------------------------------------------------------------+-------------+----------+--------------------------+--------------------------+
|Customer_HK                                                     |Customer_Name|Is_Current|Effective_From            |Effective_To              |
+----------------------------------------------------------------+-------------+----------+--------------------------+--------------------------+
|a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|Customer_1   |false     |2026-07-13 10:01:07.445381|2026-07-13 14:14:28.987952|
+----------------------------------------------------------------+-------------+----------+--------------------------+--------------------------+



In [0]:
new_current_records = changed_records.select(
    "Customer_HK",
    "Customer_Name",
    "Email",
    "City",
    "HashDiff",
    "Effective_From",
    "Effective_To",
    "Is_Current",
    "Record_Source"
)

In [0]:
new_current_records.show(truncate=False)

+----------------------------------------------------------------+-------------+----------------+---------+----------------------------------------------------------------+--------------------------+------------+----------+-------------+
|Customer_HK                                                     |Customer_Name|Email           |City     |HashDiff                                                        |Effective_From            |Effective_To|Is_Current|Record_Source|
+----------------------------------------------------------------+-------------+----------------+---------+----------------------------------------------------------------+--------------------------+------------+----------+-------------+
|a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|John Smith   |john@example.com|Hyderabad|d86317901a2a5c524cd50d9901c40d495316615513cb78142c18d5ae79243142|2026-07-13 14:15:22.958566|NULL        |true      |Incremental  |
|NULL                                           

In [0]:
changed_keys = changed_records.select("Customer_HK").distinct()

unchanged_records = current_sat.join(
    changed_keys,
    on="Customer_HK",
    how="left_anti"
)

In [0]:
unchanged_records.show(5, truncate=False)

+----------------------------------------------------------------+-------------+-------------------+------+----------------------------------------------------------------+--------------------------+-------------+--------------------------+------------+----------+
|Customer_HK                                                     |Customer_Name|Email              |City  |HashDiff                                                        |Load_Date                 |Record_Source|Effective_From            |Effective_To|Is_Current|
+----------------------------------------------------------------+-------------+-------------------+------+----------------------------------------------------------------+--------------------------+-------------+--------------------------+------------+----------+
|fdc98571fc93e8482954dcc8034bda27f571bd691cd86510a849dce99759933d|Customer_2   |customer2@gmail.com|City_3|5682cc4efd8ffa1d19650189ea009de35fbc251a30732ce3f70d412f20d561b0|2026-07-13 10:01:07.445381|Custom

In [0]:
final_sat_customer = (
    unchanged_records
        .unionByName(closed_records)
        .unionByName(new_current_records)
)

In [0]:
current_sat.printSchema()

root
 |-- Customer_HK: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- HashDiff: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Effective_From: timestamp (nullable = true)
 |-- Effective_To: timestamp (nullable = true)
 |-- Is_Current: boolean (nullable = true)



In [0]:
closed_records.printSchema()

root
 |-- Customer_HK: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- HashDiff: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Effective_From: timestamp (nullable = true)
 |-- Effective_To: timestamp (nullable = false)
 |-- Is_Current: boolean (nullable = false)



In [0]:
new_current_records.printSchema()

root
 |-- Customer_HK: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- HashDiff: string (nullable = true)
 |-- Effective_From: timestamp (nullable = false)
 |-- Effective_To: timestamp (nullable = true)
 |-- Is_Current: boolean (nullable = false)
 |-- Record_Source: string (nullable = false)



In [0]:
print(current_sat.columns)
print(closed_records.columns)
print(new_current_records.columns)

['Customer_HK', 'Customer_Name', 'Email', 'City', 'HashDiff', 'Record_Source', 'Effective_From', 'Effective_To', 'Is_Current']
['Customer_HK', 'Customer_Name', 'Email', 'City', 'HashDiff', 'Record_Source', 'Effective_From', 'Effective_To', 'Is_Current']
['Customer_HK', 'Customer_Name', 'Email', 'City', 'HashDiff', 'Effective_From', 'Effective_To', 'Is_Current', 'Record_Source']


In [0]:
current_sat = current_sat.drop("Load_Date")
closed_records = closed_records.drop("Load_Date")

In [0]:
print(current_sat.columns)
print(closed_records.columns)
print(new_current_records.columns)

['Customer_HK', 'Customer_Name', 'Email', 'City', 'HashDiff', 'Record_Source', 'Effective_From', 'Effective_To', 'Is_Current']
['Customer_HK', 'Customer_Name', 'Email', 'City', 'HashDiff', 'Record_Source', 'Effective_From', 'Effective_To', 'Is_Current']
['Customer_HK', 'Customer_Name', 'Email', 'City', 'HashDiff', 'Effective_From', 'Effective_To', 'Is_Current', 'Record_Source']


In [0]:
changed_keys = changed_records.select("Customer_HK").distinct()

unchanged_records = current_sat.join(
    changed_keys,
    on="Customer_HK",
    how="left_anti"
)

In [0]:
final_sat_customer = (
    unchanged_records
    .unionByName(closed_records)
    .unionByName(new_current_records)
)

In [0]:
%sql
DROP TABLE IF EXISTS default.sat_customer_scd2;

In [0]:
final_sat_customer.write \
    .mode("overwrite") \
    .saveAsTable("default.sat_customer_scd2")

In [0]:
%sql
SELECT *
FROM default.sat_customer_scd2;

Customer_HK,Customer_Name,Email,City,HashDiff,Record_Source,Effective_From,Effective_To,Is_Current
fdc98571fc93e8482954dcc8034bda27f571bd691cd86510a849dce99759933d,Customer_2,customer2@gmail.com,City_3,5682cc4efd8ffa1d19650189ea009de35fbc251a30732ce3f70d412f20d561b0,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
64604487c74ef2fb0aca58ed3e951aacc1f537baf96c023b75441012e4b92880,Customer_3,customer3@gmail.com,City_4,add7d8c2cae5b60736a253725ef6c4c6caf7094d03344f6929bbdc3a73916c29,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
f12ce50422ff0b232d51ad5764fad492b2f6bc3f1a74f3e5c04cc5d2742f9605,Customer_4,customer4@gmail.com,City_5,4c3f69d853c686b5fedeae7dc9819954cfd38eb267ce8d3e5f723983e03edb24,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
95adbabc0d9bd895d140d98d75be9ad39f4d91cc7a53b37cfdfac32f5b75a332,Customer_5,customer5@gmail.com,City_6,0aa7ca5592698827074d663c0a58cc6fcb33b23827708dfc6f0944cbf4767f4f,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
2db7a28cc4284c070736249ec3394b310cf755e62e74fc5693557c409618d9bc,Customer_6,customer6@gmail.com,City_7,36f30d2ec95ba3df7e3c525bc242290f57467b3404a649759bdd76d39156da18,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
ac8d2a6d24e8a318136589c9cfd4684116f2ca05714ca7c377be40929db6de5a,Customer_7,customer7@gmail.com,City_8,adebb4f2206fc22c75c14c24e68cf4563b3c7d914f3abecfa866d9814d479910,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
8d9e24cf98ed83b57da04302c28fb6d126f58019e590d2b8783b7ea38e546933,Customer_8,customer8@gmail.com,City_9,c68a4386969e864d73e6070597c07b76efb1bbf6ac57c6cbead30861760b6ea8,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
de36046be14c2817aceb8feb35714eaf11449ff97e5e986cb01888b78989ae3b,Customer_9,customer9@gmail.com,City_10,7ff8b515de624a41b3ef66c9b7c309b83422cb794f8c372eaa9845402f5e7253,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
e5f231360af6b3cc1ea64abb14e6549a75ca2d9fa6a531df7f9c3ad7633671fc,Customer_10,customer10@gmail.com,City_1,44a1e835ddfdf97eef8043df10f48211a98fb016dc49153de559bb081c1c6082,Customer_CSV,2026-07-13T10:01:07.445Z,null,true
89870e378593b983856ff246ed01a2b6533e07d8017cae8b6992d8b6c461af5b,Customer_11,customer11@gmail.com,City_2,d33e60e228a7327bba937aa32a445ba5d64f58af1369017439f95aad748bdee1,Customer_CSV,2026-07-13T10:01:07.445Z,null,true


In [0]:
%sql
SELECT
    Customer_HK,
    Customer_Name,
    City,
    Effective_From,
    Effective_To,
    Is_Current
FROM default.sat_customer_scd2
ORDER BY Customer_HK, Effective_From;

Customer_HK,Customer_Name,City,Effective_From,Effective_To,Is_Current
null,Rahul,Bangalore,2026-07-13T14:30:13.317Z,null,true
006e2a65a73cbfa2e07e91605e070297dd5e66bb9864e1f03933d648504f772d,Customer_75,City_6,2026-07-13T10:01:07.445Z,null,true
07baa9263b762e56124ab05efeca811044b40c629d78642ff30698d995d66ded,Customer_52,City_3,2026-07-13T10:01:07.445Z,null,true
0c0f5e68069384cf122abdbe9d3ab96b94e2ea49ea9baf7d9e97672406ecd265,Customer_44,City_5,2026-07-13T10:01:07.445Z,null,true
0c3e56106144e48b5db0e3e7509bef2976781f27b149c04c02a107ff3b9f28ce,Customer_16,City_7,2026-07-13T10:01:07.445Z,null,true
0c7cd539bb306a407f3af834c4f37bbc113b1d31d110b954f4d821a63ee8643d,Customer_20,City_1,2026-07-13T10:01:07.445Z,null,true
0de0e365fe160633be65abf4524fc8b9d97751eeca5c79047df940ea7ce3c9ca,Customer_93,City_4,2026-07-13T10:01:07.445Z,null,true
12c9d576fcf1e674f778502559f3d84a3e25cb28d062f1c51b591434161d462d,Customer_55,City_6,2026-07-13T10:01:07.445Z,null,true
12cc10f1e70da8efba21795f5339b0b115c70899a6f38366b422a9acab43d327,Customer_85,City_6,2026-07-13T10:01:07.445Z,null,true
16b2578dfb67c9069a5b6b07c1a29bfbea2006668e2006df5e97ca9558734cfd,Customer_65,City_6,2026-07-13T10:01:07.445Z,null,true
